In [1]:
%load_ext autoreload
%autoreload 2

from jaref_bot.data.http_api import ExchangeManager, BybitRestAPI, OKXRestAPI, GateIORestAPI
from jaref_bot.db.db_manager import DBManager
from config import host, user, password, db_name

import pandas as pd
import numpy as np
import logging
from datetime import datetime
from time import sleep
from decimal import Decimal, ROUND_DOWN

from asyncio.exceptions import CancelledError
import aiohttp, asyncio
import nest_asyncio
nest_asyncio.apply()

In [2]:
market_fees = {'bybit_spot': 0.0018, 'bybit_linear': 0.001, 'okx_spot': 0.001, 'okx_linear': 0.0005,
               'gate_spot': 0.002, 'gate_linear': 0.0005}

In [3]:
def get_volume(orderbook, volume):
    total_usdt = 0
    total_vol = 0

    for price, curr_vol in orderbook:
        if total_vol < volume:
            if curr_vol < volume - total_vol:
                usdt_amount = curr_vol * price
                total_usdt += usdt_amount
                total_vol += curr_vol
            else:
                for _ in range(int(curr_vol)):
                    if total_vol < volume:
                        total_usdt += price
                        total_vol += 1
                    else:
                        break
                break
    return total_vol, total_usdt, total_usdt / total_vol

In [4]:
async def close_order(long_exc, short_exc, symbol, volume):
    """
    :param long_exc: биржа, по которой была открыта лонговая сделка.
    :param short_exc: биржа, по которой была открыта шортовая сделка.
    :param symbol: название токена вида 'ADA', 'BTC', 'XLM' etc.
    :param volume: объём в базовой валюте
    """
    exc_dict = {'bybit': BybitRestAPI, 'okx': OKXRestAPI, 'gate': GateIORestAPI}
    exchange1, market_type1 = long_exc.split('_')
    exchange2, market_type2 = short_exc.split('_')

    exc_manager = ExchangeManager()
    exc_manager.add_market(long_exc, exc_dict[exchange1](market_type1))
    exc_manager.add_market(short_exc, exc_dict[exchange2](market_type2))

    orders = await exc_manager.get_orderbook(symbol=symbol, limit=20)
    # Биржа, по которой была открыта лонговая сделка. Теперь на ней мы продаём токены.
    long_vol, long_usdt, long_avg_price = get_volume(orders[long_exc]['bid'], volume=volume)

    # Биржа, по которой была открыта шортовая сделка. Теперь на ней мы покупаем.
    short_vol, short_usdt, short_avg_price = get_volume(orders[short_exc]['ask'], volume=volume)

    # long order
    db_manager.close_order(token=symbol, exchange=exchange1, market_type=market_type1,
                close_price=long_avg_price, close_avg_price=long_avg_price, close_usdt_amount=long_usdt,
                qty=long_vol, close_fee=market_fees[long_exc])
    
    # short order
    db_manager.close_order(token=symbol, exchange=exchange2, market_type=market_type2,
                close_price=short_avg_price, close_avg_price=short_avg_price, close_usdt_amount=short_usdt,
                qty=short_vol, close_fee=market_fees[short_exc])

    print(f"Close order. Sell {long_vol} {symbol} for {long_usdt} with fee {market_fees[long_exc]}")
    print(f"Close order. Buy {short_vol} {symbol} for {short_usdt} with fee {market_fees[short_exc]}")

In [5]:
db_params = {'host': host, 'user': user, 'password': password, 'database': db_name}
db_manager = DBManager(db_params)

In [ ]:
edge = 0
while True:
    try:
        print(f'{datetime.now().strftime('%H:%M:%S')}', end='\r')
        # Таблица с текущими ценами
        current_orders = db_manager.get_table('current_orders')
        
        # Таблица с текущими открытыми ордерами
        current_data = db_manager.get_table('current_data')
        selected_orders = current_orders[['token', 'exchange', 'market_type', 'order_side']].copy()
        selected_orders['token'] = selected_orders['token'] + '_USDT'
        
        # Отбираем из current_data только те строки, в которых есть цены по текущим открытым ордерам
        df_out = current_data.merge(selected_orders, on=['token', 'exchange', 'market_type'], how='inner')
    
        # Только если таблица с ордерами не пустая
        if not df_out.empty:
            # Группируем и считаем diff
            agg = df_out.pivot(index="token", columns="order_side", values=["ask_price", "bid_price"])
            agg.columns = [f"{col[0]}_{col[1]}" for col in agg.columns]
            agg = agg.reset_index()
            agg["diff"] = (agg["bid_price_buy"] / agg["ask_price_sell"] - 1) * 100
            
            # Формируем список на закрытие ордеров
            list_to_close = agg[agg["diff"] > edge]['token'].tolist()
            list_to_close = [x.split('_')[0] for x in list_to_close]
            
            for token in list_to_close:
                print(token)
                tdf = current_orders[current_orders['token'] == token]
                print(tdf[['exchange', 'market_type', 'order_side', 'qty']])
                for i, row in tdf.iterrows():
                    if row['order_side'] == 'buy':
                        short_exc = row['exchange'] + '_' + row['market_type']
                    elif row['order_side'] == 'sell':
                        long_exc = row['exchange'] + '_' + row['market_type']
                    volume = row['qty']
                print(f'{long_exc=} {short_exc=}')
                await close_order(long_exc=long_exc, short_exc=short_exc, symbol=token, volume=volume)
    
        sleep(0.5)
    except KeyboardInterrupt:
        print('Завершение работы.')
        break

In [ ]:
# Текущие цены
df_out[df_out['token'] == 'MNT_USDT']

In [ ]:
(1.0439 / 1.0523 - 1) * 100

In [ ]:
# Закупочные цены
current_orders[current_orders['token'] == 'MNT']

In [ ]:
# Open order. Buy 94 MNT for 98.9068 with fee 0.0005
# Open order. Sell 94 MNT for 99.2358 with fee 0.001

In [ ]:
# Закупился
(1.0557 / 1.0522 - 1) * 100